In [1]:
import numpy as np
import pandas as pd
import yfinance as yf
from Black_Scholes import Black_Scholes_Call, bs_vega_wrapper
from scipy.optimize import newton, brentq
from datetime import datetime, timedelta, timezone
import matplotlib.pyplot as plt

In [2]:
ticker = "SPY"

In [3]:
# ============================================================
# 1. Download SPY data (past 1 year)
# ============================================================
data = yf.download(
    ticker,
    period="1y",
    interval="1d",
    auto_adjust=True,
    progress=False
)

ticker_var = yf.Ticker(ticker)

if data.empty:
    raise RuntimeError("SPY data download failed")

# Use close prices as 1D numpy array
S = pd.DataFrame(data[["Close"]].to_numpy(dtype=float))
N = len(S)
print(S)


              0
0    576.994263
1    570.164490
2    576.292480
3    566.062683
4    569.235352
..          ...
246  687.349976
247  693.150024
248  689.299988
249  685.989990
250  686.380005

[251 rows x 1 columns]


In [4]:
# ============================================================
# 2. Strike price (ATM, chosen by construction)
# ============================================================

# Real options data slightly out of the money find the option price closest to asset price
# Fix the trading window
#K = float(S.iloc[N-1].values[0])
#print(K)

In [5]:
# ============================================================
# 3. Time domain (years)
# ============================================================
T = 60.0 / 365.0  # one month in years
t = np.linspace(0.0, T, N, dtype=float)

In [6]:
# ============================================================
# 6. Risk-free rate 
# ============================================================
# Fetch historical data for 13-week T-Bill
t_bill = yf.Ticker("^IRX")
# Get the latest closing price
risk_free_rate = t_bill.history(period="1d")['Close'].iloc[-1]

# Get the decimal from % value 
r = risk_free_rate/100


In [ ]:
# ============================================================
# 7. Implied Volatility (IV)
# ============================================================
#window_size = 21
#sigma_data = S.pct_change().rolling(window_size).std()*(252**0.5)
#print(sigma_data)
#sigma = float(sigma_data.iloc[250])

# Change to 252 day rolling window
# generate sigma from market price of options back solve this becomes input to the PINN
# can analyze the greeks
# Can use volatility surface sigma(S,K)

exp_dates = pd.to_datetime(ticker_var.options)

# target ≈ 2 months out
target_date = pd.Timestamp.today() + pd.DateOffset(months=2)

# nearest expiration
exp = exp_dates[np.abs(exp_dates - target_date).argmin()]
exp = exp.strftime("%Y-%m-%d")

opt_chain = ticker_var.option_chain(exp)

S_close = S.iloc[-1, 0]


df = opt_chain.calls


otm_options = df[df['strike'] >= S_close]
otm_options = otm_options[otm_options['lastPrice'] > 1]
selected_option = otm_options.iloc[0]

K = float(selected_option['strike'])
C_market = float(selected_option['lastPrice'])
print("Selected OTM Option:")
print(K)
print(C_market)




def iv_obj(sigma, S, K, r, T, C_market):
    return Black_Scholes_Call(S, K, T, r, sigma) - C_market

plotiv = []
for sigma in np.linspace(-10,10,1000):
    plotiv.append(iv_obj(sigma, S_close, K, r, T, C_market))

plt.figure()
plt.plot(plotiv)
plt.grid()
plt.show()
iv = brentq(iv_obj, -2.0, 2.0, args=(S_close, K, r, T, C_market))

#def get_iv(S, K, r, T, C_market, x0=0.2):
#    return newton(iv_obj, x0=x0, fprime=bs_vega_wrapper, args=(S, K, r, T, C_market))

#iv = get_iv(S_close, K, r, T, C_market)
print(f"Implied Volatility: {iv}")
    


AttributeError: 'TimedeltaIndex' object has no attribute 'abs'

In [ ]:
# ============================================================
# 4. Normalized price input
# ============================================================
x = S / K
# PINN input tensor: (x, t)
X_pinn = np.column_stack((x, t))

In [ ]:
# ============================================================
# 7. PINN spatial domain bounds 
# ============================================================
S0 = float(S.iloc[-1,0])
n_std = 3.0  # 3-sigma bound

S_max = S0 * np.exp(n_std * iv * np.sqrt(T))
S_min = S0 * np.exp(-n_std * iv * np.sqrt(T))

x_max = S_max / K
x_min = S_min / K

In [ ]:
# ============================================================
# 8. Output summary
# ============================================================
print("========== PINN DATA SUMMARY ==========")
print(f"Strike K              : {K}")
print(f"Time domain [years]   : [0.0, {T}]")
print(f"Volatility sigma      : {iv}")
print(f"Risk-free rate r      : {r}")
print(f"S_min, S_max          : {S_min}, {S_max}")
print(f"x_min, x_max          : {x_min}, {x_max}")
print(f"Time to expiry        : {T}")
print("=======================================")